# Celebal Technologies – Week 6 Assignment

## Spark Architecture and Performance Optimization using PySpark

### Submitted By

**Akshak Gupta**  
**Domain:** Data Engineering

# Creating Spark Session

The Spark Session is the entry point of every Spark application.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder \
    .appName("Celebal Week 6 Assignment") \
    .getOrCreate()

In [0]:
df = spark.read.csv(
    "/path/to/Sample_-_Superstore.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

# Demonstrating Lazy Evaluation

Spark builds a logical plan when transformations are defined — nothing runs until an action is triggered.

In [0]:
lazy_df = (
    df.filter(col("Category") == "Technology")
    .select(
        "Order ID",
        "Customer Name",
        "Product Name",
        "Profit"
    )
    .withColumnRenamed(
        "Profit",
        "Net Profit"
    )
)

At this point Spark has only built the execution plan.

No data has actually been read or processed — no Action has been called yet.

In [0]:
lazy_df.show(5)

+--------------+------------------+--------------------+----------+
|      Order ID|     Customer Name|        Product Name|Net Profit|
+--------------+------------------+--------------------+----------+
|CA-2014-115812|   Brosina Hoffman|Mitel 5320 IP Pho...|   90.7152|
|CA-2014-115812|   Brosina Hoffman|Konftel 250 Confe...|   68.3568|
|CA-2014-143336|Zuschuss Donatelli|Cisco SPA 501G IP...|    16.011|
|CA-2016-121755|     Eric Hoffmann|Imation 8GB Mini ...|    9.5997|
|CA-2016-117590|         Gene Hale|         GE 30524EE4|   54.8772|
+--------------+------------------+--------------------+----------+
only showing top 5 rows


The `show()` function is an Action — this is what triggers Spark to actually execute the plan and return results.

In [0]:
print("Total Records :", lazy_df.count())

Total Records : 1847


## CSV vs Parquet

In [0]:
df = spark.read.csv(
    "/path/to/Sample_-_Superstore.csv",
    header=True,
    inferSchema=True
)

Reading data from CSV format.

In [0]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [0]:
print("Total Records :", df.count())

Total Records : 9994


In [0]:
print("Total Columns :", len(df.columns))

Total Columns : 21


In [0]:
df.dtypes

[('Row ID', 'int'),
 ('Order ID', 'string'),
 ('Order Date', 'string'),
 ('Ship Date', 'string'),
 ('Ship Mode', 'string'),
 ('Customer ID', 'string'),
 ('Customer Name', 'string'),
 ('Segment', 'string'),
 ('Country', 'string'),
 ('City', 'string'),
 ('State', 'string'),
 ('Postal Code', 'int'),
 ('Region', 'string'),
 ('Product ID', 'string'),
 ('Category', 'string'),
 ('Sub-Category', 'string'),
 ('Product Name', 'string'),
 ('Sales', 'string'),
 ('Quantity', 'string'),
 ('Discount', 'string'),
 ('Profit', 'double')]

In [0]:
df.describe().show()

+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|summary|            Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|    City|  State|       Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|  count|              9994|          9994|   

# Parquet

Parquet is a columnar storage format designed for big data processing. It stores schema inside the file and compresses far better than CSV.

In [0]:
df.write.mode("overwrite").parquet(
    "/path/to/output/superstore_parquet"
)

In [0]:
parquet_df = spark.read.parquet(
    "/path/to/output/superstore_parquet"
)

In [0]:
parquet_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
parquet_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
parquet_df.select(
"Customer Name",
"Sales"
).show()

+------------------+--------+
|     Customer Name|   Sales|
+------------------+--------+
|       Claire Gute|  261.96|
|       Claire Gute|  731.94|
|   Darrin Van Huff|   14.62|
|    Sean O'Donnell|957.5775|
|    Sean O'Donnell|  22.368|
|   Brosina Hoffman|   48.86|
|   Brosina Hoffman|    7.28|
|   Brosina Hoffman| 907.152|
|   Brosina Hoffman|  18.504|
|   Brosina Hoffman|   114.9|
|   Brosina Hoffman|1706.184|
|   Brosina Hoffman| 911.424|
|      Andrew Allen|  15.552|
|      Irene Maddox| 407.976|
|     Harold Pawlan|   68.81|
|     Harold Pawlan|   2.544|
|         Pete Kriz|  665.88|
|   Alejandro Grove|    55.5|
|Zuschuss Donatelli|    8.56|
|Zuschuss Donatelli|  213.48|
+------------------+--------+
only showing top 20 rows


### Selecting Specific Columns

In [0]:
selected_df = df.select(
    "Order ID",
    "Customer Name",
    "Category",
    "Sales",
    "Profit"
)
selected_df.show(5)

+--------------+---------------+---------------+--------+--------+
|      Order ID|  Customer Name|       Category|   Sales|  Profit|
+--------------+---------------+---------------+--------+--------+
|CA-2016-152156|    Claire Gute|      Furniture|  261.96| 41.9136|
|CA-2016-152156|    Claire Gute|      Furniture|  731.94| 219.582|
|CA-2016-138688|Darrin Van Huff|Office Supplies|   14.62|  6.8714|
|US-2015-108966| Sean O'Donnell|      Furniture|957.5775|-383.031|
|US-2015-108966| Sean O'Donnell|Office Supplies|  22.368|  2.5164|
+--------------+---------------+---------------+--------+--------+
only showing top 5 rows


In [0]:
technology_df = df.filter(
    col("Category") == "Technology"
).select(
    "Order ID",
    "Product Name",
    "Sales"
)
technology_df.show(10)

+--------------+--------------------+--------+
|      Order ID|        Product Name|   Sales|
+--------------+--------------------+--------+
|CA-2014-115812|Mitel 5320 IP Pho...| 907.152|
|CA-2014-115812|Konftel 250 Confe...| 911.424|
|CA-2014-143336|Cisco SPA 501G IP...|  213.48|
|CA-2016-121755|Imation 8GB Mini ...|   90.57|
|CA-2016-117590|         GE 30524EE4|1097.544|
|CA-2015-117415|Plantronics HL10 ...| 371.168|
|CA-2017-120999|  Panasonic Kx-TS550| 147.168|
|CA-2016-118255|Verbatim 25 GB 6x...|   45.98|
|CA-2016-169194|Imation 8GB Micro...|      45|
|CA-2016-169194|LF Elite 3D Dazzl...|    21.8|
+--------------+--------------------+--------+
only showing top 10 rows


In [0]:
west_df = df.filter(
    col("Region") == "West"
)
west_df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|Darrin Van Huff|Corporate|United States|Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|

In [0]:
high_profit = df.filter(
    (col("Region") == "West") &
    (col("Profit") > 100)
)
high_profit.show()

+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+---------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|    Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+---------+--------+--------+----------+
|    14|CA-2016-161389| 12/5/2016|12/10/2016|Standard Class|   IM-15070|        Irene Maddox|   Consumer|United States|      Seattle|Washington|      98103|  West|OFF-BI-10003656|Office Supplie

In [0]:
filtered_df = df.filter(
    (col("Region") == "West") |
    (col("Category") == "Technology")
)
filtered_df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|   Darrin Van Huff|  Corporate|United States|    Los Angeles|    California|      90036|  West|OFF-LA-10000240|O

### Renaming Columns

In [0]:
df_transform = df.withColumnRenamed(
    "Sales",
    "Total Sales"
)

In [0]:
df_transform = df_transform.withColumnRenamed(
    "Profit",
    "Net Profit"
)

In [0]:
df_transform.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Total Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Net Profit: double (nullable = true)



### Casting Data Types

In [0]:
df_transform = df_transform.withColumn(
    "Net Profit",
    col("Net Profit").cast("Double")
)

In [0]:
df_transform.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Total Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Net Profit: double (nullable = true)



### Creating New Columns

In [0]:
df_transform = df_transform.withColumn(
    "Profit Margin",
    round(
        (col("Net Profit") / col("Net Profit")) * 10,
        2
    )
)

In [0]:
df_transform = df_transform.withColumn(
    "Profit Flag",
    when(col("Net Profit") > 0, "Profitable").otherwise("Loss")
)

In [0]:
df_transform.select(
    "Total Sales",
    "Net Profit",
    "Profit Margin",
    "Profit Flag"
).show(10)

+-----------+----------+-------------+-----------+
|Total Sales|Net Profit|Profit Margin|Profit Flag|
+-----------+----------+-------------+-----------+
|     261.96|   41.9136|         10.0|Profitable |
|     731.94|   219.582|         10.0|Profitable |
|      14.62|    6.8714|         10.0|Profitable |
|   957.5775|  -383.031|        -10.0|       Loss|
|     22.368|    2.5164|         10.0|Profitable |
|      48.86|   14.1694|         10.0|Profitable |
|       7.28|    1.9656|         10.0|Profitable |
|    907.152|   90.7152|         10.0|Profitable |
|     18.504|    5.7825|         10.0|Profitable |
|      114.9|     34.47|         10.0|Profitable |
+-----------+----------+-------------+-----------+
only showing top 10 rows


In [0]:
df_transform.select(
    "Order ID",
    "Customer Name",
    "Product Name",
    "Region",
    "Category",
    "Total Sales",
    "Net Profit",
    "Profit Flag"
).show(10)

+--------------+---------------+--------------------+------+---------------+-----------+----------+-----------+
|      Order ID|  Customer Name|        Product Name|Region|       Category|Total Sales|Net Profit|Profit Flag|
+--------------+---------------+--------------------+------+---------------+-----------+----------+-----------+
|CA-2016-152156|    Claire Gute|Bush Somerset Col...| South|      Furniture|     261.96|   41.9136|Profitable |
|CA-2016-152156|    Claire Gute|Hon Deluxe Fabric...| South|      Furniture|     731.94|   219.582|Profitable |
|CA-2016-138688|Darrin Van Huff|Self-Adhesive Add...|  West|Office Supplies|      14.62|    6.8714|Profitable |
|US-2015-108966| Sean O'Donnell|Bretford CR4500 S...| South|      Furniture|   957.5775|  -383.031|       Loss|
|US-2015-108966| Sean O'Donnell|Eldon Fold 'N Rol...| South|Office Supplies|     22.368|    2.5164|Profitable |
|CA-2014-115812|Brosina Hoffman|Eldon Expressions...|  West|      Furniture|      48.86|   14.1694|Profi

### Sorting the Dataset

In [0]:
df_transform.orderBy(
    col("Net Profit").desc()
).show(10)

+------+--------------+----------+---------+--------------+-----------+--------------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+-------------+-----------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|Total Sales|Quantity|Discount|Net Profit|Profit Margin|Profit Flag|
+------+--------------+----------+---------+--------------+-----------+--------------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+-------------+-----------+
|  6827|CA-2016-118689| 10/2/2016|10/9/2016|Standard Class|   TC-20980|        Tamara Chand|  Corporate

# Performance Optimization in Apache Spark

In [0]:
sales_by_region = df_transform.groupBy(
    "Region"
).agg(
    round(sum("Net Profit"), 2).alias("Total Profit"),
    count("Order ID").alias("Order Count")
)
sales_by_region.show()

+-------+------------+-----------+
| Region|Total Profit|Order Count|
+-------+------------+-----------+
|  South|    46650.34|       1620|
|Central|     40150.5|       2323|
|   East|    91603.06|       2848|
|   West|    107303.7|       3203|
+-------+------------+-----------+


In [0]:
sales_by_region.orderBy(
    col("Total Profit").desc()
).show()

+-------+------------+-----------+
| Region|Total Profit|Order Count|
+-------+------------+-----------+
|   West|    107303.7|       3203|
|   East|    91603.06|       2848|
|  South|    46650.34|       1620|
|Central|     40150.5|       2323|
+-------+------------+-----------+


In [0]:
parquet_df.filter(
    col("Region") == "West"
).show()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|Darrin Van Huff|Corporate|United States|Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|

In [0]:
from pyspark.sql.functions import when, count
df_transform.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in ["Order ID","Customer Name","Region","Category","Net Profit","Discount"]
]).show()

+--------+-------------+------+--------+----------+--------+
|Order ID|Customer Name|Region|Category|Net Profit|Discount|
+--------+-------------+------+--------+----------+--------+
|       0|            0|     0|       0|         0|       0|
+--------+-------------+------+--------+----------+--------+


In [0]:
efficient_df = df_transform.filter(
    col("Net Profit") > 500
)
efficient_df.show(10)

+------+--------------+----------+---------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+-------------+-----------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|Total Sales|Quantity|Discount|Net Profit|Profit Margin|Profit Flag|
+------+--------------+----------+---------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+-------------+-----------+
|   150|CA-2016-114489| 12/5/2016|12/9/2016|Standard Class|   JE-16165|      Justin Ellison|  Corporate|Unite

In [0]:
df_transform.select(
    "Customer Name",
    "Product Name",

).show()

+------------------+--------------------+
|     Customer Name|        Product Name|
+------------------+--------------------+
|       Claire Gute|Bush Somerset Col...|
|       Claire Gute|Hon Deluxe Fabric...|
|   Darrin Van Huff|Self-Adhesive Add...|
|    Sean O'Donnell|Bretford CR4500 S...|
|    Sean O'Donnell|Eldon Fold 'N Rol...|
|   Brosina Hoffman|Eldon Expressions...|
|   Brosina Hoffman|          Newell 322|
|   Brosina Hoffman|Mitel 5320 IP Pho...|
|   Brosina Hoffman|DXL Angle-View Bi...|
|   Brosina Hoffman|Belkin F5C206VTEL...|
|   Brosina Hoffman|Chromcraft Rectan...|
|   Brosina Hoffman|Konftel 250 Confe...|
|      Andrew Allen|          Xerox 1967|
|      Irene Maddox|Fellowes PB200 Pl...|
|     Harold Pawlan|Holmes Replacemen...|
|     Harold Pawlan|Storex DuraTech R...|
|         Pete Kriz|Stur-D-Stor Shelv...|
|   Alejandro Grove|Fellowes Super St...|
|Zuschuss Donatelli|          Newell 341|
|Zuschuss Donatelli|Cisco SPA 501G IP...|
+------------------+--------------

In [0]:
df_transform.filter(
    col("Net Profit") > 500
).select(
    "Customer Name",
    "Net Profit"
).explain()

== Physical Plan ==
*(1) Project [Customer Name#6, Profit#20 AS Net Profit#23]
+- *(1) Filter (isnotnull(Profit#20) AND (Profit#20 > 500.0))
   +- *(1) ColumnarToRow
      +- FileScan parquet [Customer Name#6,Profit#20] Batched: true, DataFilters: [isnotnull(Profit#20), (Profit#20 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[...superstore_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(Profit), GreaterThan(Profit,500.0)], ReadSchema: struct<Customer Name:string,Profit:double>




# Building an End-to-End Spark Data Pipeline

In [0]:
pipeline_df = spark.read.csv(
    "/path/to/Sample_-_Superstore.csv",
    header=True,
    inferSchema=True
)

In [0]:
pipeline_df = pipeline_df.fillna({
    "Discount": 0
})

In [0]:
pipeline_df = pipeline_df.withColumnRenamed(
    "Sales",
    "Total Sales"
)
pipeline_df = pipeline_df.withColumnRenamed(
    "Profit",
    "Net Profit"
)

In [0]:
pipeline_df = pipeline_df.withColumn(
    "Net Profit",
    col("Net Profit").cast("Double")
)

In [0]:
pipeline_df = pipeline_df.withColumn(
    "Profit Flag",
    when(col("Net Profit") > 0, "Profitable").otherwise("Loss")
)

In [0]:
pipeline_df = pipeline_df.filter(
    (col("Region") == "West") &
    (col("Net Profit") > 0)
)

In [0]:
pipeline_df = pipeline_df.select(
    "Order ID",
    "Customer Name",
    "Region",
    "Category",
    "Product Name",
    "Quantity",
    "Total Sales",
    "Net Profit",
    "Profit Flag"
)

In [0]:
pipeline_df.show(10)

+--------------+---------------+------+---------------+--------------------+--------+-----------+----------+-----------+
|      Order ID|  Customer Name|Region|       Category|        Product Name|Quantity|Total Sales|Net Profit|Profit Flag|
+--------------+---------------+------+---------------+--------------------+--------+-----------+----------+-----------+
|CA-2016-138688|Darrin Van Huff|  West|Office Supplies|Self-Adhesive Add...|       2|      14.62|    6.8714| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|      Furniture|Eldon Expressions...|       7|      48.86|   14.1694| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|Office Supplies|          Newell 322|       4|       7.28|    1.9656| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|     Technology|Mitel 5320 IP Pho...|       6|    907.152|   90.7152| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|Office Supplies|DXL Angle-View Bi...|       3|     18.504|    5.7825| Profitable|
|CA-2014-115812|Brosina Hoffman|

The complete Spark pipeline has successfully loaded, transformed, filtered, and prepared the dataset for further analysis or storage.

In [0]:
pipeline_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/path/to/output/processed_csv")

In [0]:
pipeline_df.write \
    .mode("overwrite") \
    .parquet("/path/to/output/processed_parquet")

In [0]:
saved_df = spark.read.parquet(
    "/path/to/output/processed_parquet"
)
saved_df.show(10)

+--------------+---------------+------+---------------+--------------------+--------+-----------+----------+-----------+
|      Order ID|  Customer Name|Region|       Category|        Product Name|Quantity|Total Sales|Net Profit|Profit Flag|
+--------------+---------------+------+---------------+--------------------+--------+-----------+----------+-----------+
|CA-2016-138688|Darrin Van Huff|  West|Office Supplies|Self-Adhesive Add...|       2|      14.62|    6.8714| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|      Furniture|Eldon Expressions...|       7|      48.86|   14.1694| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|Office Supplies|          Newell 322|       4|       7.28|    1.9656| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|     Technology|Mitel 5320 IP Pho...|       6|    907.152|   90.7152| Profitable|
|CA-2014-115812|Brosina Hoffman|  West|Office Supplies|DXL Angle-View Bi...|       3|     18.504|    5.7825| Profitable|
|CA-2014-115812|Brosina Hoffman|